# Week 7: Advanced Models
- Gradient Boosting
- Hyperparameter tuning (depth, learning rate, n_estimators)

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.impute import SimpleImputer

#Using LightGBM
import lightgbm as lgb

# Validation
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

In [5]:
# Load Dataset
df = pd.read_csv("/content/drive/MyDrive/IDX Data Science/with_features.csv")

/tmp/ipykernel_6073/3800747513.py:2: DtypeWarning: Columns (0,1,4,64,79,81,82,83,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/IDX Data Science/with_features.csv")


Previous Model Results:

In [6]:
import pandas as pd

# Constructing an updated comparison table for the Test Set metrics with the correct post-FE values
comparison_data = {
    "Model": [
        "Linear/Ridge Regression", "Linear/Ridge Regression",
        "Decision Tree", "Decision Tree",
        "Random Forest", "Random Forest"
    ],
    "Feature Stage": [
        "Before FE (Baseline OLS)", "After FE (Ridge CV Auto-Alpha)",
        "Before FE", "After FE",
        "Before FE (Optimized)", "After FE"
    ],
    "R² Score": [0.7573, 0.8592, 0.7849, 0.8771, 0.8598, 0.9020],
    "MAE ($)": [298593.64, 238845.19, 222519.74, 219944.07, 188239.03, 192208.63],
    "MdAPE (%)": [19.63, 12.38, 10.00, 10.71, 9.07, 8.93]
}

comparison_df = pd.DataFrame(comparison_data)

# Formatting columns for better presentation
comparison_df["R² Score"] = comparison_df["R² Score"].map(lambda x: f"{x:.4f}")
comparison_df["MAE ($)"] = comparison_df["MAE ($)"].map(lambda x: f"${x:,.2f}")
comparison_df["MdAPE (%)"] = comparison_df["MdAPE (%)"].map(lambda x: f"{x:.2f}%")

print("=== TEST SET MODEL PERFORMANCE COMPARISON ===")
display(comparison_df)

=== TEST SET MODEL PERFORMANCE COMPARISON ===


,Model,Feature Stage,R² Score,MAE ($),MdAPE (%)
0,Linear/Ridge Regression,Before FE (Baseline OLS),0.7573,"$298,593.64",19.63%
1,Linear/Ridge Regression,After FE (Ridge CV Auto-Alpha),0.8592,"$238,845.19",12.38%
2,Decision Tree,Before FE,0.7849,"$222,519.74",10.00%
3,Decision Tree,After FE,0.8771,"$219,944.07",10.71%
4,Random Forest,Before FE (Optimized),0.8598,"$188,239.03",9.07%
5,Random Forest,After FE,0.9020,"$192,208.63",8.93%


In [7]:
test = df[df['FileDate'] == '2026-04-01']
print(f"Test set size: {len(test)}")
validation = df[df['FileDate'] == '2026-03-01']
print(f"Validation set size: {len(validation)}")
train = df[df['FileDate'] < '2026-03-01']
print(f"Train set size: {len(train)}")

print(f"Total Rows Check: {len(test) + len(validation) + len(train)}")

Test set size: 15225
Validation set size: 14118
Train set size: 349995
Total Rows Check: 379338


In [8]:
# set outlier cutoffs based on the training set
lower_bound = train['ClosePrice'].quantile(0.05)
upper_bound = train['ClosePrice'].quantile(0.995)
print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")

Lower bound: 365000.0
Upper bound: 8150554.99999944


In [9]:
cat_cols = ['City', 'CountyOrParish', 'DistrictName']
num_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'Stories',
                'BedBathRatio', 'LotAreaRatio', 'AgeOfProperty', 'RelativeSize']
# create new copies of train, test, and validation
train = train[(train['ClosePrice'] > lower_bound) & (train['ClosePrice'] < upper_bound)]
validation = validation[(validation['ClosePrice'] > lower_bound) & (validation['ClosePrice'] < upper_bound)]
test = test[(test['ClosePrice'] > lower_bound) & (test['ClosePrice'] < upper_bound)]

# keep feature columns
train = train[cat_cols + num_cols + ['ClosePrice']]
validation = validation[cat_cols + num_cols + ['ClosePrice']]
test = test[cat_cols + num_cols + ['ClosePrice']]

In [10]:
# separate X's and Y for data sets
Y_train = train['ClosePrice'].copy()
X_train = train[num_cols + cat_cols].copy()

Y_val = validation['ClosePrice'].copy()
X_val = validation[num_cols + cat_cols].copy()

X_train_val = pd.concat([X_train, X_val], axis=0)
Y_train_val = pd.concat([Y_train, Y_val], axis=0)

Y_test = test['ClosePrice'].copy()
X_test = test[num_cols + cat_cols].copy()

In [11]:
# MdAPE
def median_absolute_percentage_error(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100
# function to evaluate different errors (R^2, MAE, MAPE, MdAPE)
def evaluate_predictions(y_true, y_pred, split_name="Validation"):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100  # Convert ratio to %
    mdape = median_absolute_percentage_error(y_true, y_pred)

    print(f"--- {split_name} Evaluation Metrics ---")
    print(f"R² Score : {r2:.4f}")
    print(f"MAE      : ${mae:,.2f}")
    print(f"MAPE     : {mape:.2f}%")
    print(f"MdAPE    : {mdape:.2f}%\n")

## Decision Tree with LightGBM

In [25]:
# Convert string/object columns to pandas category dtype in our split datasets
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [13]:
# Gemini-assisted
gbdt_model = lgb.LGBMRegressor(
    boosting_type='gbdt',       # Default: Sequential gradient boosting
    n_estimators=100,           # Number of decision trees
    learning_rate=0.05,         # Step size shrinkage
    num_leaves=31,              # Maximum leaves per tree (controls complexity)
    max_depth=-1,               # Unlimited depth (constrained by num_leaves)
    subsample=1.0,              # Row bagging fraction
    colsample_bytree=1.0,       # Feature column subsampling fraction
    random_state=42,
    n_jobs=-1,                  # Use all available CPU threads
    verbose=-1
)

In [14]:
# Validation
gbdt_model.fit(X_train, Y_train)
gbdt_val_preds = gbdt_model.predict(X_val)
evaluate_predictions(Y_val, gbdt_val_preds, split_name="Validation - LightGBM Decision Tree")

--- Validation - LightGBM Decision Tree Evaluation Metrics ---
R² Score : 0.8652
MAE      : $186,686.42
MAPE     : 14.21%
MdAPE    : 9.95%



In [26]:
# Cast categorical columns to category dtype for X_train_val
for col in cat_cols:
    X_train_val[col] = X_train_val[col].astype('category')

# Testing
gbdt_model.fit(X_train_val, Y_train_val)
gbdt_test_preds = gbdt_model.predict(X_test)
evaluate_predictions(Y_test, gbdt_test_preds, split_name="Test - LightGBM Decision Tree")

--- Test - LightGBM Decision Tree Evaluation Metrics ---
R² Score : 0.8781
MAE      : $177,812.89
MAPE     : 12.80%
MdAPE    : 8.53%



In [16]:
import pandas as pd

# Data from comparison_df for 'Decision Tree' 'After FE':
# R² Score: 0.8771, MAE: $219,944.07, MdAPE: 10.71%
# Data from our LightGBM runs:
# Validation - LightGBM: R²: 0.8652, MAE: $186,686.42, MdAPE: 9.95%
# Test - LightGBM:       R²: 0.8633, MAE: $193,572.14, MdAPE: 10.05%

results_comparison = {
    "Model/Evaluation Split": [
        "Original Decision Tree (After FE) - Test Set",
        "LightGBM Regressor - Validation Set",
        "LightGBM Regressor - Test Set"
    ],
    "R² Score": [0.8771, 0.8652, 0.8633],
    "MAE ($)": [219944.07, 186686.42, 193572.14],
    "MdAPE (%)": [10.71, 9.95, 10.05]
}

comparison_table = pd.DataFrame(results_comparison)

# Format the columns for visualization
comparison_formatted = comparison_table.copy()
comparison_formatted["R² Score"] = comparison_formatted["R² Score"].map(lambda x: f"{x:.4f}")
comparison_formatted["MAE ($)"] = comparison_formatted["MAE ($)"].map(lambda x: f"${x:,.2f}")
comparison_formatted["MdAPE (%)"] = comparison_formatted["MdAPE (%)"].map(lambda x: f"{x:.2f}%")

print("=== PERFORMANCE COMPARISON: ORIGINAL DECISION TREE VS. LIGHTGBM ===")
display(comparison_formatted)

=== PERFORMANCE COMPARISON: ORIGINAL DECISION TREE VS. LIGHTGBM ===


,Model/Evaluation Split,R² Score,MAE ($),MdAPE (%)
0,Original Decision Tree (After FE) - Test Set,0.8771,"$219,944.07",10.71%
1,LightGBM Regressor - Validation Set,0.8652,"$186,686.42",9.95%
2,LightGBM Regressor - Test Set,0.8633,"$193,572.14",10.05%


While R^2 remains mostly similar across the original and gradient-boosted decision tree model, there is a moderate decrease in MAE and slight decrease in MdAPE for the Gradient Boosted model

## Random Forest with LightGBM

In [17]:
# ---------------------------------------------------------
# LightGBM as a Random Forest
# ---------------------------------------------------------
gbrf_model = lgb.LGBMRegressor(
    boosting_type='rf',         # Switch boosting algorithm to Random Forest
    n_estimators=100,           # Number of parallel decision trees in forest
    num_leaves=31,              # Capacity per tree
    subsample=0.8,              # Bagging fraction (rows) - REQUIRED for RF
    subsample_freq=1,           # MUST be set > 0 (1 = resample rows every tree)
    colsample_bytree=0.8,       # Feature subsampling per tree (rand forest feature split)
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [18]:
gbrf_model.fit(X_train, Y_train)
gbrf_val_preds = gbrf_model.predict(X_val)
evaluate_predictions(Y_val, gbrf_val_preds, split_name="Validation - LightGBM Random Forest")


--- Validation - LightGBM Random Forest Evaluation Metrics ---
R² Score : 0.8070
MAE      : $237,130.19
MAPE     : 19.15%
MdAPE    : 14.31%



In [19]:
# Testing
gbrf_model.fit(X_train_val, Y_train_val)
gbrf_test_preds = gbrf_model.predict(X_test)
evaluate_predictions(Y_test, gbrf_test_preds, split_name="Test - LightGBM Random Forest")

--- Test - LightGBM Random Forest Evaluation Metrics ---
R² Score : 0.8041
MAE      : $245,395.07
MAPE     : 19.34%
MdAPE    : 14.43%



In [20]:
import pandas as pd

# Original Random Forest metrics from comparison_df (After FE):
# After FE:             R² = 0.9020, MAE = $192,208.63, MdAPE = 8.93%
#
# LightGBM Random Forest metrics from current runs:
# Validation:           R² = 0.8070, MAE = $237,130.19, MdAPE = 14.31%
# Test:                 R² = 0.8041, MAE = $245,395.07, MdAPE = 14.43%

rf_comparison_data = {
    "Model/Evaluation Split": [
        "Original RF (After FE) - Test Set",
        "LightGBM RF - Validation Set",
        "LightGBM RF - Test Set"
    ],
    "R² Score": [0.9020, 0.8070, 0.8041],
    "MAE ($)": [192208.63, 237130.19, 245395.07],
    "MdAPE (%)": [8.93, 14.31, 14.43]
}

rf_comparison_df = pd.DataFrame(rf_comparison_data)

# Formatting columns
rf_comparison_formatted = rf_comparison_df.copy()
rf_comparison_formatted["R² Score"] = rf_comparison_formatted["R² Score"].map(lambda x: f"{x:.4f}")
rf_comparison_formatted["MAE ($)"] = rf_comparison_formatted["MAE ($)"].map(lambda x: f"${x:,.2f}")
rf_comparison_formatted["MdAPE (%)"] = rf_comparison_formatted["MdAPE (%)"].map(lambda x: f"{x:.2f}%")

print("=== RANDOM FOREST MODELS PERFORMANCE COMPARISON ===")
display(rf_comparison_formatted)

=== RANDOM FOREST MODELS PERFORMANCE COMPARISON ===


,Model/Evaluation Split,R² Score,MAE ($),MdAPE (%)
0,Original RF (After FE) - Test Set,0.9020,"$192,208.63",8.93%
1,LightGBM RF - Validation Set,0.8070,"$237,130.19",14.31%
2,LightGBM RF - Test Set,0.8041,"$245,395.07",14.43%


In the Random Forest model, the Gradient boosted version performs worse than the original, meaning that there is probably some work to be done for hyperparameter tuning.

## HyperParameter Tuning

## Decision Tree

In [21]:
from sklearn.model_selection import ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Helper to replace infinite values with NaN in our splits
for df_split in [X_train, X_val, X_train_val, X_test]:
    df_split.replace([np.inf, -np.inf], np.nan, inplace=True)

# Configure a preprocessor that imputes numerical values and target-encodes categoricals
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', TargetEncoder(random_state=42), cat_cols)
    ]
)

dt_grid = {
    'max_depth': [3, 5, 8, 12, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

best_val_score_dt = float('inf')
best_params_dt = None

print("=== TUNING DECISION TREE (dt) ===")
for params in ParameterGrid(dt_grid):
    # Create a pipeline combining preprocessing and estimator
    dt_model = DecisionTreeRegressor(**params, random_state=42)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', dt_model)
    ])

    # Fit on training split
    pipeline.fit(X_train, Y_train)

    # Evaluate on validation split
    val_preds = pipeline.predict(X_val)
    val_score = mean_absolute_error(Y_val, val_preds)

    if val_score < best_val_score_dt:
        best_val_score_dt = val_score
        best_params_dt = params

# Retrain best configuration on combined Train + Validation
final_dt_model = DecisionTreeRegressor(**best_params_dt, random_state=42)
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', final_dt_model)
])
final_pipeline.fit(X_train_val, Y_train_val)

# Final test evaluation
dt_test_preds = final_pipeline.predict(X_test)
evaluate_predictions(Y_test, dt_test_preds, split_name=f"TEST SET | DT Best Params: {best_params_dt}")

=== TUNING DECISION TREE (dt) ===
--- TEST SET | DT Best Params: {'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 2} Evaluation Metrics ---
R² Score : 0.8307
MAE      : $211,204.09
MAPE     : 15.13%
MdAPE    : 10.42%



## GB Decision Tree

In [22]:
# LightGBM
gbdt_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 8],
    'num_leaves': [15, 31], 40, 50, 60,
}

best_val_score_gbdt = float('inf')
best_params_gbdt = None

print("=== TUNING GRADIENT BOOSTED DECISION TREE (gbdt) ===")
for params in ParameterGrid(gbdt_grid):
    gbdt_model = lgb.LGBMRegressor(
        boosting_type='gbdt',
        **params,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    # Fit on training split
    gbdt_model.fit(X_train, Y_train)

    # Evaluate on validation split
    val_preds = gbdt_model.predict(X_val)

    # Print details using the evaluation function
    #evaluate_predictions(Y_val, val_preds, f"Val | GBDT | {params}")

    # Compute MAE explicitly for hyperparameter comparison
    val_score = mean_absolute_error(Y_val, val_preds)

    if val_score < best_val_score_gbdt:
        best_val_score_gbdt = val_score
        best_params_gbdt = params

# Retrain best configuration on combined Train + Validation
final_gbdt = lgb.LGBMRegressor(
    boosting_type='gbdt',
    **best_params_gbdt,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
final_gbdt.fit(X_train_val, Y_train_val)

# Final test evaluation
gbdt_test_preds = final_gbdt.predict(X_test)
evaluate_predictions(Y_test, gbdt_test_preds, f"TEST SET | GBDT Best Params: {best_params_gbdt}")

=== TUNING GRADIENT BOOSTED DECISION TREE (gbdt) ===
--- TEST SET | GBDT Best Params: {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 300, 'num_leaves': 31} Evaluation Metrics ---
R² Score : 0.8781
MAE      : $177,812.89
MAPE     : 12.80%
MdAPE    : 8.53%



## Random Forest

In [41]:
import numpy as np
import pandas as pd
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
import lightgbm as lgb

# 1. Re-verify PredefinedSplit alignment
test_fold = np.concatenate([
    -1 * np.ones(len(X_train), dtype=int),
     0 * np.ones(len(X_val), dtype=int)
])
ps = PredefinedSplit(test_fold=test_fold)

# 2. Focused structural search grid (8 candidate combinations instead of 108)
gbrf_grid = {
    'learning_rate': [0.5, 0.8, 1.0],         # Standard RF is 1.0; < 1.0 shrinks individual tree contributions
    'max_depth': [6, 8, 10, 12, 15],          # Expanded tree depths
    'num_leaves': [15, 31, 63, 127],          # Expanded leaf capacities
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8],                  # Row sampling (bagging fraction)
    'colsample_bytree': [0.7, 0.8]            # Column sampling (feature fraction)
}

# 3. FAST BASE ESTIMATOR: Set n_jobs=1 inside LGBMRegressor to stop thread thrashing!
fast_gbrf_base = lgb.LGBMRegressor(
    boosting_type='rf',
    learning_rate=1.0,
    subsample=0.8,              # Fixed bagging fraction
    subsample_freq=1,           # Resample every iteration
    colsample_bytree=0.8,       # Fixed feature fraction
    random_state=42,
    n_jobs=1,                   # CRITICAL: Set to 1 so GridSearchCV handles outer parallelism
    verbose=-1
)

# 4. Randomized Search with outer parallelism (n_jobs=-1)
gbrf_search = RandomizedSearchCV(
    estimator=fast_gbrf_base,
    param_distributions=gbrf_grid,
    n_iter=15,
    cv=ps,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1                   # Evaluates parameter sets across CPU cores in parallel
)

# Run fast search
gbrf_search.fit(X_train_val, Y_train_val)

# 5. Fit final production model on winning parameters
best_gbrf_params = gbrf_search.best_params_

final_gbrf = lgb.LGBMRegressor(
    boosting_type='rf',
    subsample_freq=1,
    **best_gbrf_params,
    random_state=42,
    n_jobs=-1,                  # Final single-model fit uses all CPU cores
    verbose=-1
)
final_gbrf.fit(X_train_val, Y_train_val)

# Evaluate on holdout test set
gbrf_preds = final_gbrf.predict(X_test)
evaluate_predictions(Y_test, gbrf_preds, f"FAST TEST SET | LightGBM RF {best_gbrf_params}")

--- FAST TEST SET | LightGBM RF {'subsample': 0.7, 'num_leaves': 127, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.5, 'colsample_bytree': 0.8} Evaluation Metrics ---
R² Score : 0.8351
MAE      : $220,079.75
MAPE     : 16.95%
MdAPE    : 12.23%



# Summary

In [42]:
import pandas as pd

# Compilation of metrics from all model stages
summary_data = {
    "Model Type": [
        "Linear/Ridge Regression (Baseline)",
        "Linear/Ridge Regression (After FE)",
        "Original Decision Tree (After FE)",
        "Tuned Decision Tree (Scikit-Learn)",
        "LightGBM GBDT (Default)",
        "Tuned LightGBM GBDT",
        "Original Random Forest (After FE)",
        "LightGBM Random Forest (Default)",
        "Tuned LightGBM Random Forest"
    ],
    "Search/Tuning Status": [
        "N/A",
        "Ridge CV Auto-Alpha",
        "Default",
        "GridSearchCV",
        "Default",
        "ParameterGrid Search",
        "Optimized",
        "Default",
        "RandomizedSearchCV"
    ],
    "R² Score": [0.7573, 0.8592, 0.8771, 0.8307, 0.8781, 0.8781, 0.9020, 0.8041, 0.8351],
    "MAE ($)": [298593.64, 238845.19, 219944.07, 211204.09, 177812.89, 177812.89, 192208.63, 245395.07, 220079.75],
    "MdAPE (%)": [19.63, 12.38, 10.71, 10.42, 8.53, 8.53, 8.93, 14.43, 12.23]
}

summary_df = pd.DataFrame(summary_data)

# Format for display
summary_formatted = summary_df.copy()
summary_formatted["R² Score"] = summary_formatted["R² Score"].map(lambda x: f"{x:.4f}")
summary_formatted["MAE ($)"] = summary_formatted["MAE ($)"].map(lambda x: f"${x:,.2f}")
summary_formatted["MdAPE (%)"] = summary_formatted["MdAPE (%)"].map(lambda x: f"{x:.2f}%")

print("=== OVERALL TEST SET PERFORMANCE SUMMARY ===")
display(summary_formatted)

=== OVERALL TEST SET PERFORMANCE SUMMARY ===


,Model Type,Search/Tuning Status,R² Score,MAE ($),MdAPE (%)
0,Linear/Ridge Regression (Baseline),N/A,0.7573,"$298,593.64",19.63%
1,Linear/Ridge Regression (After FE),Ridge CV Auto-Alpha,0.8592,"$238,845.19",12.38%
2,Original Decision Tree (After FE),Default,0.8771,"$219,944.07",10.71%
3,Tuned Decision Tree (Scikit-Learn),GridSearchCV,0.8307,"$211,204.09",10.42%
4,LightGBM GBDT (Default),Default,0.8781,"$177,812.89",8.53%
5,Tuned LightGBM GBDT,ParameterGrid Search,0.8781,"$177,812.89",8.53%
6,Original Random Forest (After FE),Optimized,0.9020,"$192,208.63",8.93%
7,LightGBM Random Forest (Default),Default,0.8041,"$245,395.07",14.43%
8,Tuned LightGBM Random Forest,RandomizedSearchCV,0.8351,"$220,079.75",12.23%


R^2 fluctuates the most within models while MAE and MdAPE typically showed improvement for the Decision Tree model. For the Random Forest model, my original feature engineer model seems to perform the best overall, with the highest R^2, 2nd lowest MAE and 2nd lowest MdAPE. The parameters I used for this model were:
n_estimators=100, max_depth=14, min_samples_leaf=15, n_jobs=-1, and random_state=42.